# OmniVoice Kit backend on Google Colab + ngrok

Notebook này clone/pull source, cài dependencies trong `backend/`, chạy FastAPI backend và expose API bằng ngrok.

Sau refactor, `pyproject.toml`, `uv.lock`, `models/`, `data/`, `assets/` nằm trong thư mục `backend/`; frontend vẫn ở root repo.


In [ ]:
# ===== Config =====
REPO_URL = "https://github.com/googlemap2/omnivoice-kit.git"
BRANCH = "main"
PROJECT_DIR = "/content/omnivoice-kit"
BACKEND_DIR = f"{PROJECT_DIR}/backend"

# Cach 1: dien token vao day.
# Cach 2: de rong va tao Colab Secret ten NGROK_AUTHTOKEN.
NGROK_AUTHTOKEN = ""

# False: model tu tai vao disk tam cua Colab moi runtime.
# True: cache models vao Google Drive de tranh tai lai.
USE_GOOGLE_DRIVE_MODELS = False
DRIVE_MODELS_DIR = "/content/drive/MyDrive/omnivoice-kit/backend/models"

API_PORT = 8000
CORS_ORIGINS = "http://localhost:3000,http://127.0.0.1:3000"


In [ ]:
import os

os.environ["VOICEKIT_DATABASE_URL"] = "postgresql://..."
os.environ["VOICEKIT_CORS_ORIGINS"] = "*"
os.environ["VOICEKIT_CACHE_EMOTION_TTS"] = "true"
os.environ["VOICEKIT_CACHE_TTS"] = "true"


## 1. Chuan bi cache model

In [ ]:
from pathlib import Path

if USE_GOOGLE_DRIVE_MODELS:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_MODELS_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Model cache dir: {DRIVE_MODELS_DIR}")
else:
    print("Models will be downloaded into backend/models on the temporary Colab disk when needed.")


## 2. Clone hoặc pull source

In [ ]:
import os
import subprocess
from pathlib import Path

project_path = Path(PROJECT_DIR)
backend_path = Path(BACKEND_DIR)

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if (project_path / ".git").exists():
    run(["git", "fetch", "origin"], cwd=project_path)
    run(["git", "checkout", BRANCH], cwd=project_path)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=project_path)
else:
    project_path.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_path)])

if not backend_path.is_dir():
    raise FileNotFoundError(f"Backend directory not found: {backend_path}")

os.chdir(backend_path)
print(f"Repo directory: {project_path}")
print(f"Backend directory: {Path.cwd()}")


## 3. Chuan bi thu muc `backend/models/`


In [ ]:
from pathlib import Path

models_path = Path(BACKEND_DIR) / "models"

if USE_GOOGLE_DRIVE_MODELS:
    drive_models_path = Path(DRIVE_MODELS_DIR)
    drive_models_path.mkdir(parents=True, exist_ok=True)
    if models_path.exists() and not models_path.is_symlink():
        print(f"backend/models already exists at {models_path}; keeping it as-is.")
    elif not models_path.exists():
        models_path.symlink_to(drive_models_path, target_is_directory=True)
        print(f"Linked {models_path} -> {drive_models_path}")
    else:
        print(f"backend/models symlink already exists: {models_path} -> {models_path.resolve()}")
else:
    models_path.mkdir(parents=True, exist_ok=True)
    print(f"Using {models_path}")


## 4. Cai dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!python -m pip install -U pip uv pyngrok

# onnxruntime 1.24.x ships cp311 wheels, so force the backend project venv to Python 3.11.
!cd {BACKEND_DIR} && uv python install 3.11
!cd {BACKEND_DIR} && uv sync --python 3.11


## 5. Chay FastAPI backend

In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

backend_path = Path(BACKEND_DIR)
os.chdir(backend_path)
os.environ["VOICEKIT_CORS_ORIGINS"] = CORS_ORIGINS

# Stop server cu neu cell nay duoc chay lai.
pid_file = backend_path / ".colab_api_pid"
if pid_file.exists():
    try:
        old_pid = int(pid_file.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
        print(f"Stopped old API process: {old_pid}")
    except Exception as exc:
        print(f"Could not stop old API process: {exc}")

log_path = backend_path / "api.log"
log_file = open(log_path, "w", encoding="utf-8")
process = subprocess.Popen(
    ["uv", "run", "uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
    cwd=backend_path,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
pid_file.write_text(str(process.pid), encoding="utf-8")
print(f"API started on port {API_PORT}. PID: {process.pid}")
print(f"Log file: {log_path}")
time.sleep(5)
!tail -n 40 {log_path}


## 6. Mo ngrok tunnel

In [ ]:
from pyngrok import ngrok

token = NGROK_AUTHTOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("NGROK_AUTHTOKEN") or ""
    except Exception:
        token = ""

if not token:
    raise ValueError("Please set NGROK_AUTHTOKEN in the config cell or Colab Secrets.")

ngrok.set_auth_token(token)
ngrok.kill()
public_url = ngrok.connect(API_PORT, "http")
print("Backend URL:", public_url.public_url)
print("Health URL:", f"{public_url.public_url}/health")

## Lenh huu ich

Xem log backend realtime. Cell này sẽ chạy liên tục; bấm stop cell khi không cần xem log nữa.

In [ ]:
!tail -n 100 -f {BACKEND_DIR}/api.log
